**Etapa 1:** Visão Geral e Proporção de Churn (Base de 80.000 Registros)

* **Objetivo**: Carregar a base de dados bancários, inspecionar o esquema e entender a proporção inicial de clientes ativos versus cancelados (Churn).

* **Ações realizadas**: 
  * Importação das bibliotecas essenciais e leitura das primeiras linhas
  * Descrição estatística das variáveis numéricas e categóricas.
  * Renomeação dos rótulos/colunas para facilitar o manuseio .
  * Contagem geral de clientes ativos e taxa de Churn


In [0]:
# Manipulação e estruturação de dados
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning e Métricas (caso vá avançar para modelagem preditiva)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [0]:
%sql
select * from workspace.default.bank_churn_dataset limit 30

In [0]:

%sql
describe workspace.default.bank_churn_dataset;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bank_churn_dataset_pt AS
SELECT 
    id                       AS id_cliente,
    full_name                AS nome_completo,
    credit_sco               AS score_credito,
    gender                   AS genero,
    age                      AS idade,
    occupation               AS profissao,
    balance                  AS saldo,
    monthly_ir               AS renda_mensal,
    address                  AS endereco,
    origin_province          AS provincia_origem,
    tenure_ye                AS anos_relacionamento,
    married                  AS casado,
    nums_card                AS num_cartoes,
    nums_service             AS num_servicos,
    active_member            AS membro_ativo,
    last_active_date         AS data_ultima_atividade,
    last_transaction_month   AS ultimo_mes_transacao,
    created_date             AS data_criacao,
    CASE WHEN exit = TRUE THEN 1 ELSE 0 END AS churn,
    customer_segment         AS segmento_cliente,
    engagement_score         AS score_engajamento,
    loyalty_level            AS nivel_fidelidade,
    digital_behavior         AS comportamento_digital,
    risk_score               AS score_risco,
    risk_segment             AS segmento_risco_texto,
    -- Transformação do segmento de risco em escala numérica
    CASE 
        WHEN risk_segment = 'Low' THEN 0
        WHEN risk_segment = 'Medium' THEN 1

        ELSE NULL 
    END                      AS segmento_risco,
    cluster_group            AS grupo_cluster
FROM workspace.default.bank_churn_dataset;

Verificando se há dados nulos em nossa base de dados

In [0]:
df_pd = spark.read.table("workspace.default.bank_churn_dataset_pt").toPandas()
df_pd.info()

Iniciando perguntas de negócio.


1 - Qual é a Taxa de Churn Geral da Base?

In [0]:
%sql
SELECT 
    churn,
    COUNT(*) AS total_clientes,
    ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()), 2) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY churn;

Databricks visualization. Run in Databricks to view.

Volume Total Analisado: 80.000 clientes cadastrados na base de dados.  

Clientes Cancelados (Churn): 14.400 clientes, o que representa 18% da base total.  


Clientes Ativos: 65.600 clientes, correspondendo a 82% do total

2 - Perfil Demográfico: Idade × Churn

In [0]:
%sql
SELECT 
    churn,
    COUNT(*) AS qtd_clientes,
    ROUND(AVG(idade), 1) AS media_idade,
    ROUND(MIN(idade), 0) AS idade_minima,
    ROUND(MAX(idade), 0) AS idade_maxima,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY churn;

Databricks visualization. Run in Databricks to view.

A diferença é de aproximadamente 2,2 anos a favor dos clientes ativos, indicando que o público que cancela o serviço tende a ser um pouco mais jovem do que a média da base geral.

In [0]:
%sql
SELECT 
    segmento_risco,
    COUNT(*) AS total_clientes,
    ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()), 2) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY segmento_risco
ORDER BY segmento_risco;

In [0]:
%sql
SELECT 
    churn,
    segmento_risco,
    COUNT(*) AS total_clientes,
    ROUND((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY churn)), 2) AS percentual_do_grupo
FROM workspace.default.bank_churn_dataset_pt
GROUP BY churn, segmento_risco
ORDER BY churn, segmento_risco;

Base Saudável: 96,7% dos clientes que continuam no banco são de baixo risco.

O Alerta (Onde está o erro?): Quase 79% dos clientes que cancelaram a conta não tinham nenhum risco registrado. Ou seja, o banco está perdendo clientes "bons" e seguros.

Oportunidade: Como esses clientes que saem não apresentam risco financeiro, eles provavelmente vão embora por falta de engajamento, atendimento ruim ou concorrência — e não por inadimplência. O foco de retenção precisa olhar para além do risco de crédito.

In [0]:
%sql
SELECT 
    nivel_fidelidade,
    membro_ativo,
    COUNT(*) AS total_clientes,
    ROUND(AVG(score_engajamento), 2) AS media_engajamento,
    ROUND(AVG(anos_relacionamento), 1) AS media_anos_relacionamento
FROM workspace.default.bank_churn_dataset_pt
WHERE churn = 1 AND segmento_risco = 0
GROUP BY nivel_fidelidade, membro_ativo
ORDER BY total_clientes DESC;

Perfil Predominante (Bronze e Inativo):
A esmagadora maioria do churn de baixo risco (10.130 clientes) concentra-se no nível Bronze com membro_ativo = false. Esse grupo apresenta um padrão claro de evasão silenciosa: entram com pouco tempo de relacionamento (média de 1,8 anos) e deixam a instituição com baixíssimo engajamento (média de 16,48).

A Oportunidade na Proposta de Valor:
O comportamento desse segmento evidencia a necessidade de uma estratégia focada em ativação precoce. Em contrapartida, clientes com níveis de fidelidade Silver e Gold praticamente não aparecem nessa base de evasão, o que comprova que quanto maior e mais sólida é a fidelização, menor é a propensão ao cancelamento.
 o volume é expressivo isoladamente (11.311 pessoas), ele representa apenas 1% da base total mas 70% de todo o churn . Portanto, esse grupo de baixo risco e inativo  é o principal motor do churn que afeta a saúde financeira ou o volume estratégico do banco.

In [0]:
%sql
SELECT 
    churn,
    ROUND(AVG(num_cartoes), 2) AS media_cartoes,
    ROUND(AVG(num_servicos), 2) AS media_servicos,
    ROUND(AVG(score_engajamento), 2) AS media_engajamento,
    ROUND(AVG(anos_relacionamento), 2) AS media_anos_relacionamento,
    COUNT(*) AS total_clientes
FROM workspace.default.bank_churn_dataset_pt
GROUP BY churn;

Databricks visualization. Run in Databricks to view.

O Poder do Ecossistema (Serviços): A diferença mais expressiva está no número de serviços utilizados. Quem permanece ativo consome em média 3,81 serviços, enquanto quem dá churn utiliza apenas 2,44. Isso confirma que quanto mais integrado o cliente está aos produtos do banco, menor a chance de evasão.

O Peso do Engajamento: Esse comportamento é fortemente respaldado pela média de engajamento, que apresenta um patamar consideravelmente superior entre os clientes ativos (34,31) em comparação aos que cancelaram (19,94).

A Leve Influência dos Cartões: Clientes que ficam possuem uma média ligeiramente maior de cartões (2,7) em relação aos que cancelam (2,32), reforçando que a variedade de cartões também atua como uma pequena barreira de saída.

O Tempo de Relacionamento: Curiosamente, a média de anos de relacionamento é muito próxima entre os dois grupos (cerca de 1,79 anos para ativos contra 1,7 anos para churn). Isso indica que o cancelamento não se restringe à fase inicial, podendo ocorrer independentemente do tempo de casa caso o cliente perca o engajamento com a instituição.

In [0]:
%sql
SELECT 
    nivel_fidelidade,
    churn,
    ROUND(AVG(num_cartoes), 2) AS media_cartoes,
    ROUND(AVG(num_servicos), 2) AS media_servicos,
    ROUND(AVG(score_engajamento), 2) AS media_engajamento,
    ROUND(AVG(anos_relacionamento), 2) AS media_anos_relacionamento,
    COUNT(*) AS total_clientes
FROM workspace.default.bank_churn_dataset_pt
GROUP BY nivel_fidelidade, churn
ORDER BY nivel_fidelidade, churn;

In [0]:
%sql
SELECT 
    nivel_fidelidade,
    segmento_risco,
    churn,
    COUNT(*) AS total_clientes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentual_da_base_total
FROM workspace.default.bank_churn_dataset_pt
WHERE nivel_fidelidade = 'Bronze'
GROUP BY nivel_fidelidade, segmento_risco, churn
ORDER BY segmento_risco, churn;

O perfil Bronze representa 87% da base total e responde por impressionantes 97% de todo o churn do banco (14.020 de um total de 14.400 cancelamentos).

Quando cruzamos esse cenário com o risco, reforçamos a tese anterior: estamos prospectando muito bem e trazendo clientes qualificados para a base, mas falhamos em fazer com que esses clientes evoluam de nível de fidelidade.

Como o nível Bronze possui as menores médias de engajamento (18.8 para churns) e de serviços utilizados (2.41), o desafio central não é apenas a aquisição, mas criar trilhas de desenvolvimento para que o cliente Bronze avance para os níveis Silver e Gold, onde o churn é residual.

O Churn não é uma questão de gênero ou de tempo de casa. (Ambos são muito parecidos nos dois grupos).


3 - Saúde Financeira: Saldo e Renda Mensal × Churn

In [0]:
%sql
select
    churn,
    round(avg(saldo),2) as media_saldo,
    round(avg(renda_mensal),2) as media_renda_mensal,
    round(avg(score_credito),1) as media_score_credito,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()) AS percentual_total
from workspace.default.bank_churn_dataset_pt
group by churn

Databricks visualization. Run in Databricks to view.

Perfil do Churn vs. Ativos há uma diferença expressiva de capital: Os clientes que ficam na instituição tendem a ser muito mais rentáveis e possuem saldos substancialmente maiores em conta ( Mesmo sem o contexto exato da moeda ou o país de origem), a proporção é o que importa aqui.

Clientes que deram churn possuem um saldo médio e uma renda mensal drasticamente menores (cerca de 1/3 do valor) em comparação aos clientes que continuam ativos.)

  O Score de Crédito também reflete isso: quem vai embora possui uma pontuação de crédito inferior (média de 653 contra 691 dos ativos).  Conclusão de Negócio: O risco de churn está fortemente concentrado em clientes com menor poder aquisitivo (menor renda e saldo) e menor score de crédito dentro desta base.

In [0]:
%sql
select
     nivel_fidelidade,
    segmento_risco,
    churn,
    round(avg(saldo),2) as media_saldo,
    round(avg(renda_mensal),2) as media_renda_mensal,
    round(avg(score_credito),1) as media_score_credito,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()) AS percentual_total
from workspace.default.bank_churn_dataset_pt
WHERE nivel_fidelidade = 'Bronze' 
GROUP BY nivel_fidelidade, segmento_risco, churn
ORDER BY segmento_risco, churn;

Observamos que o grande volume da base
(76,64% dos clientes, com saldo médio de 6.651.557 e score de 692) permanece ativo no baixo risco (segmento_risco = 0 e churn = 0).

No entanto, o ponto de atenção crítico reside nos clientes de baixo risco que evadem (segmento_risco = 0 e churn = 1), que representam 15,79% da base com saldo médio de 2.186.228 e score de 666,7. Já nos grupos classificados com risco mais alto (segmento_risco = 1), tanto os ativos quanto os que cancelam apresentam uma queda progressiva na renda e no escore de crédito (com pontuações médias de 601,9 e 604,6, respectivamente). A conclusão de negócio é que o risco de evasão está fortemente concentrado nos extremos de menor capital e menor escore dentro da base Bronze, reforçando a necessidade de ações direcionadas a esse público..


In [0]:
%sql
SELECT 
    membro_ativo,
    churn,
    COUNT(*) AS total_clientes,
    round((COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()),0) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY membro_ativo, churn
ORDER BY membro_ativo, churn;

Databricks visualization. Run in Databricks to view.

A esmagadora maioria dos cancelamentos vem do grupo de membros inativos (13.283 clientes, o equivalente a quase 92% de todo o churn da base).

O Poder dos Clientes Engajados: Ser um "membro ativo" é a melhor vacina contra o cancelamento. Quase ninguém que usa o banco com frequência decide ir embora (apenas 1,40% cancelaram).

Insighs:
Em vez de gastar energia tentando salvar todo mundo, o foco deve ser criar campanhas rápidas para acordar quem está com a conta parada antes que fechem a conta de vez.


O Churn é uma questão de Engajamento e Renda/Saldo. (Quem cancela usa menos serviços, tem menos saldo e está inativo)

Será que o churn está em regiões específicas?

In [0]:
%sql
SELECT 
   provincia_origem,
    COUNT(*) AS total_clientes,
    SUM(churn) AS total_churn,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS taxa_churn_pct
FROM workspace.default.bank_churn_dataset_pt
GROUP BY provincia_origem
ORDER BY taxa_churn_pct DESC;

Databricks visualization. Run in Databricks to view.

Análise por Província (provincia_origem): A taxa de churn oscila muito próxima da média geral na maioria das regiões (entre 18% e 20%), sendo ligeiramente menor em TP. Hồ Chí Minh (17,15%) e destacando-se negativamente apenas no grupo genérico Tỉnh khác (outras províncias, com 20,17%) ,ao analisar os dados de forma média não nos entrega um alerta suficiente para focarmos em uma praça específica, mas sim indica que o problema se comporta de forma padronizada, independentemente da região.

Será que o churn está emum segmento específico?

In [0]:
%sql
SELECT 
    segmento_cliente,
    COUNT(CASE WHEN churn = 1 THEN 1 END) AS total_churn_segmento,
    (SELECT COUNT(*) FROM workspace.default.bank_churn_dataset_pt WHERE churn = 1) AS total_churn_geral,
    (COUNT(CASE WHEN churn = 1 THEN 1 END) * 100.0 / (SELECT COUNT(*) FROM workspace.default.bank_churn_dataset_pt WHERE churn = 1)) AS pct_representacao_no_churn_total
FROM workspace.default.bank_churn_dataset_pt
where nivel_fidelidade = "Bronze"
GROUP BY segmento_cliente
ORDER BY total_churn_segmento DESC;

Concentração Crítica (97,5% do Problema): Quase a totalidade do churn do banco é um fenômeno concentrado na base da pirâmide (Mass + Emerging). O ralo de clientes não está espalhado de forma homogênea.

Preservação do Alto Valor: Os segmentos de alta renda (Affluent e Priority) somam apenas 358 cancelamentos no total (2,49%). Isso demonstra que o banco não está perdendo seus clientes de maior rentabilidade/patrimônio de forma expressiva.

In [0]:
%sql
SELECT 
    segmento_cliente,
    COUNT(*) AS total_clientes,
    SUM(churn) AS total_churn,
    ROUND(SUM(churn) * 100.0 / COUNT(*), 2) AS taxa_churn_pct
FROM workspace.default.bank_churn_dataset_pt
GROUP BY segmento_cliente
ORDER BY taxa_churn_pct DESC;

O verdadeiro calcanhar de Aquiles do banco está aqui. O segmento Mass (Massa) apresenta uma taxa de churn alarmante de 39,48%, concentrando o maior volume de cancelamentos da base. Em contrapartida, os segmentos de maior valor (Affluent e Priority) possuem taxas quase nulas (3,10% e 0,07%).

Quase 40% dos clientes do segmento Mass estão abandonando o banco. Isso indica que a oferta de valor para o público de massa pode ser pouco atraente, 

Direcionamento Estratégico: A esforço do time de CRM, inteligência de dados e produto deve focar 100% na régua de relacionamento, onboarding e ativação dos segmentos Mass e Emerging, pois resolver a dormência desses dois grupos estanca a perda de clientes da instituição.

Plano de Ação Sugerido:

Criar uma força-tarefa de retenção focada exclusivamente no segmento Mass.

Investigar o grupo Tỉnh khác (Outras províncias) para entender se a falta de atendimento físico ou suporte digital regional está acelerando o cancelamento fora dos grandes centros.